In [ ]:
# Makes it so any variable or statement on it's own line gets printed w/o print()
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Start with importing the packages
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import scipy
from scipy import stats
import altair as alt
import scipy.spatial as sp, scipy.cluster.hierarchy as hc

#pip install psynlig
#from psynlig import plot_correlation_heatmap


# Setting some visualization preferences
pd.set_option('display.precision', 2)
pd.set_option('display.max_columns',10)
#plt.style.use('seaborn-talk')

# We're also going to tell Jupyter to use inline plotting instead of notebook plotting
# It basically means you don't have to use plt.show() in every cell
%matplotlib inline



### Importing our expression dataset
We're going to start by importing data from the melanoma spreadsheet we worked with last week, so that we've got something to work with.

In [ ]:
# importing the melanoma dataset
melanoma_log2 = pd.read_excel('~/LECTURE_MATERIALS/DataFiles/melanoma_zerosRemoved_log2transformed_2026.xlsx',index_col = 0)
melanoma_log2.head()

In [ ]:
# importing the DESeq2 analysis output
DESeqInfo = pd.read_csv('~/LECTURE_MATERIALS/DataFiles/deseq_results_melanoma.csv',index_col = 0)
DESeqInfo.head()

### Prepping the data

We want to start with extracting the log2 count expression data.

In [ ]:
melanoma_log2_exp = melanoma_log2.loc[:,'A1BG':]
melanoma_log2_exp.head()

Out first heatmap example will be to examine the pairwise Pearson correlation values between each sample.

In [ ]:
samplecorr = melanoma_log2_exp.transpose().corr(method='pearson')
samplecorr

In [ ]:
samplecorr.rename_axis(None, axis=1,inplace=True)
samplecorr

In [ ]:
samplecorrUnstacked = samplecorr.unstack()
samplecorrUnstacked = samplecorrUnstacked.reset_index()
samplecorrUnstacked = samplecorrUnstacked.set_axis(['Sample1','Sample2','PearsonsR'], axis=1)
samplecorrUnstacked.head()

### Heatmaps in Altair

In [ ]:
#default heatmap
alt.Chart(samplecorrUnstacked).mark_rect().encode(
    x='Sample1', #N for nominal
    y='Sample2',
    color='PearsonsR' #Q for quantitative
    )

In [ ]:
# Custom bins, different color scheme, custom legend
alt.Chart(samplecorrUnstacked).mark_rect().encode(
    x='Sample1', 
    y='Sample2',
    #color='PearsonsR' 
    color=alt.Color('PearsonsR',
                    scale=alt.Scale(scheme='magma'),
                    bin=alt.Bin(maxbins=10),
                    legend=alt.Legend(title='Pearson Correlation (binned)')
                    ),
    )

In [ ]:
# Adding an extra layer of text
# Configure common options
base = alt.Chart(samplecorrUnstacked.round(2)).encode(
        x='Sample1',
        y='Sample2'
)

# Configure heatmap
heatmap = base.mark_rect().encode(
    color=alt.Color('PearsonsR',
        scale=alt.Scale(scheme='magma'),
        bin=alt.Bin(maxbins=10),
        legend=alt.Legend(title='Pearson Correlation (binned)')  
    )
)

# Configure text
text = base.mark_text(baseline='middle',fontSize = 8).encode(
    text='PearsonsR',
    color=alt.condition(
        alt.datum.PearsonsR > 0.95,
        alt.value('black'),
        alt.value('white')
    )
)

# Draw the chart
heatmap + text

### Heatmaps in Seaborn

If you don't care about interactive plotting, you can also use Seaborn to generate a similar heatmap with a single line of code.

In [ ]:
# plot a heatmap with annotation
sns.heatmap(samplecorr, annot=True, annot_kws={"size": 7},cmap = sns.color_palette("magma", 10),vmin = 0.8, vmax = 1.0);

# Clustermaps in Seaborn


In addition to generating a heatmap visualization, Seaborn clustermaps also use hierarchical clustering to group samples and genes that are similar to each other. 

To demonstrate this, we're going to use a clustermap to visualize the gene expression of genes that are significantly differentially expressed.

In [ ]:
# Identifying genes of interest
GenesOfInterestAbs = DESeqInfo[(DESeqInfo.padj < 0.01) & (abs(DESeqInfo.log2FoldChange) > 3.3)].index.tolist()
GenesOfInterest2 = list(set(GenesOfInterestAbs) & set(melanoma_log2_exp.columns.tolist()))
len(GenesOfInterest2)

In [ ]:
# Extracting the log2 expression values of these genes only
log2Exp_GOIabs = melanoma_log2_exp.loc[:,GenesOfInterest2]
log2Exp_GOIabs.head()

In [ ]:
# The Default clustermap
sns.clustermap(log2Exp_GOIabs);

In this default version, it's hard to see the differences between samples for the genes (the baseline expression value differences get emphasized). It's also not obvious which samples are which which category unless you read the sample labels.

#### Plotting log2 fold change

In [ ]:
FMmean_abs = np.mean(log2Exp_GOIabs.iloc[:3,:],axis = 0)
FMmean_abs

In [ ]:
log2Exp_GOIabs_diffFM = log2Exp_GOIabs - FMmean_abs
log2Exp_GOIabs_diffFM

The color palette above is not ideal for showcasing log2 fold-change. Ideally for something that can go up and down, we'd want a diverging color palette.

In [ ]:
# The Close-to-Default clustermap
sns.clustermap(log2Exp_GOIabs_diffFM,cmap = 'coolwarm');

#### Customizing visualizations: adding row color annotations

Ok, so now we'd like add a visual guide to the sample clusters by introducing a colored bar in the rows denoting the different cell types.  We'll start by isolating the the cell line info from our combined expr/meta data frame.


In [ ]:
cell_line = melanoma_log2.loc[:,'cell_line']
print(cell_line)

Next, let's visualize the basic seaborn color palette

In [ ]:
sns.palplot(sns.color_palette())

Information about other color palettes area available [here](https://seaborn.pydata.org/tutorial/color_palettes.html). For custom color palettes, go [here](https://coolors.co/palettes/trending). To set a custom color bar, look at the tutorial [here](https://medium.datadriveninvestor.com/creating-a-discrete-colorbar-with-custom-bin-sizes-in-matplotlib-50b0daf8dd46).

Now, we can assign colors from this palette to the unique cell lines in this study.

we'll create a new dictionary associating each cell line with a different color from the Seaborn default color palette.

In [ ]:
# find the set of unique cell line names.
cellLineNames = set(melanoma_log2.loc[:,'cell_line'])

In [ ]:
# dictionarry associating cell line name to a color designation
cellColor = dict(zip(cellLineNames,sns.color_palette()))
print(cellColor)

In [ ]:
# mapping the cell line colors to the individual samples
row_colors1 = cell_line.map(cellColor)
print(row_colors1)

In [ ]:
# Create clustermap
g = sns.clustermap(log2Exp_GOIabs_diffFM, row_colors=row_colors1,cmap = 'coolwarm');

You can set multiple `row_color` annotations. This [link](https://stackoverflow.com/questions/48173798/additional-row-colors-in-seaborn-cluster-map) shows you can do this.

#### Customizing visualizations: customizing the color palette, and labels

We can also manually set discrete bins for color palettes.

In [ ]:
colors = sns.color_palette("coolwarm", 7)
colors

In [ ]:
bins = [-15 , -10 , -5 , -1 , 1, 5, 10, 15]

In [ ]:
cmap = mcolors.ListedColormap(colors)
norm = mcolors.BoundaryNorm(boundaries=bins, ncolors=len(cmap.colors) )

In [ ]:
fig, ax = plt.subplots(figsize=(6, 1))
fig.subplots_adjust(bottom=0.5)
cb2 = matplotlib.colorbar.ColorbarBase(ax, cmap=cmap,
                                norm=norm,
                                boundaries= bins,
                                extend='both',
                                ticks=bins,
                                spacing='proportional',
                                orientation='horizontal')
cb2.set_label('log2 fold change')
plt.show()

In [ ]:
# Create clustermap
g = sns.clustermap(log2Exp_GOIabs_diffFM, row_colors=row_colors1 ,cmap = cmap,vmin = -15, vmax = 15,xticklabels=False);

# Manually add colorbar
cax = plt.axes(g.cbar_pos)
#cb = plt.colorbar(g.ax_heatmap.collections[0], cax=cax, ticks=bins)
cb2 = matplotlib.colorbar.ColorbarBase(cax, cmap=cmap,
                                norm=norm,
                                boundaries= bins,
                                ticks=[-15,-10,-5,5,10,15],
                                spacing='proportional',
                                orientation='vertical');
#cb2.ax.set_yticklabels(bins)
cb2.set_label('log2 fold change')
#ax = g.ax_heatmap
#ax.set_xlabel('')
plt.show()

#### Customizing visualization: choose your own clustering approach.

We spent an awful lot of time talking about distance metrics and linkage methods - and I stressed that the outcomes downstream can vary based on these decisions.  

The seaborn clustermap function allows you some customization by adjusting the `metric` and `method` arguments to adjust the distance and the linkage options, respectively. However, these options can sometimes be limiting. For maximum flexibility, you can perform own distance and linkage separately, and ask the clustermap function to incorporate that information into the resulting visualization. 

To illustrate how to do this, let's draw dendrograms based on Pearson correlation.

In [ ]:
samplecorr

In [ ]:
samplecorr_dissimilarity = 1 - abs(samplecorr)
samplecorr_dissimilarity

In [ ]:
sampleLinkage = hc.linkage(sp.distance.squareform(samplecorr_dissimilarity), method='average')

In [ ]:
threshold = 0.08
hc.dendrogram(sampleLinkage, labels=samplecorr_dissimilarity.columns, leaf_rotation=90,color_threshold = threshold);


You can also output information about which cluster each sample belongs to, based on the threshold you pick:

In [ ]:
# Clusterize the data
SampleClusterLabels = hc.fcluster(sampleLinkage, threshold, criterion='distance')

# Show the cluster
sampleClusterInfo = pd.DataFrame({'samples': melanoma_log2.index,'cluster_labels': SampleClusterLabels})
sampleClusterInfo

Let's see what happens to the colors when we toggle the `color_threshold` parameter.

Now, to tie the clustering information back into the clustergram, you can simply specify the `sampleLinkage` as the linkage information in the rows using the `row_linkage` argument.

In [ ]:
# Create clustermap
g = sns.clustermap(log2Exp_GOIabs_diffFM, row_colors=row_colors1 ,cmap = cmap,vmin = -15, vmax = 15,xticklabels=False,row_linkage = sampleLinkage);

# Manually add colorbar
cax = plt.axes(g.cbar_pos)
#cb = plt.colorbar(g.ax_heatmap.collections[0], cax=cax, ticks=bins)
cb2 = matplotlib.colorbar.ColorbarBase(cax, cmap=cmap,
                                norm=norm,
                                boundaries= bins,
                                ticks=[-15,-10,-5,5,10,15],
                                spacing='proportional',
                                orientation='vertical');
#cb2.ax.set_yticklabels(bins)
cb2.set_label('log2 fold change')
#ax = g.ax_heatmap
#ax.set_xlabel('')
plt.show()

### <font color=brown>Hands on practice</font>

Using what we've learned so far, modify the clustermap in the following ways:

1. Only include genes with positive log2 fold change, as calculated by DESeq2
2. Use Pearson correlation to inform the dendrograms on both the genes and the samples. Use a threshold cutoff of 0.6 for samples and 0.1 for genes when drawing the dendrograms using `hc.dendrogram()`, and also report the cluster labels for each gene and sample when these thresholds are used.


#### <font color = blue> Optional Bonus: heatmaps with psynlig

In [ ]:
pip install psynlig

In [ ]:
from psynlig import plot_correlation_heatmap

In [ ]:
# Code to set some keyword arguments
kwargs = {
    'text': {
        'fontsize': 'large',
    },
    'heatmap': {
        'vmin': 0.8, #lowest number visualized
        'vmax': 1, #highest number visualized 
        'cmap': 'viridis', #color map used
    },
    'figure': {
        'figsize': (8, 8) # figure size
    },
}

# Code to plot the heatmap
plot_correlation_heatmap(melanoma_log2_exp.transpose(), textcolors=['white', 'black'], **kwargs);
plt.show()